# 7. Modelos de Regressão


Este notebook explora diferentes modelos de regressão para análise de dados, incluindo Regressão Linear e Lasso, utilizando técnicas de pré-processamento como transformação Box-Cox e seleção de variáveis. O objetivo é avaliar o desempenho dos modelos em termos de métricas como RMSE e R², tanto nos datasets completos quanto após a remoção de outliers das questões do Enem (vetorizadas).


In [1]:
# Importando Dependências para Modelos de Regressão
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LassoCV

In [ ]:
# Leitura dos dados tokenizados do ENEM
enem_data = pd.read_pickle("../data/final/complete_enem_data.pkl")
enem_data.head()

,numero_questao,enunciado,alternativas,gabarito,questao,pc_amostra_acertos,ano,gabarito_texto,distratores,enunciado_tokens,...,enunciado_embbedings_word2vec_50,gabarito_embbedings_word2vec_50,distratores_embbedings_word2vec_50,similaridade_enunciado_gabarito_50,similaridade_enunciado_distratores_50,similaridade_gabarito_distratores_50,resposta_deepseek,acerto_deepseek,resposta_llamma,acerto_llamma
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,1,0.93,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,...,"[0.08906648, 0.18746194, 0.09568582, -0.018923...","[0.177859, 0.14451149, -0.033945624, 0.12138, ...","[0.16039853, 0.15997498, 0.05087997, 0.0837478...",0.728683,0.867395,0.862642,C,True,C,True
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2,0.40,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,...,"[0.0029705844, -0.0016135853, 0.007862832, 0.0...","[0.09892542, 0.057256144, 0.0030154246, 0.0359...","[0.07989774, 0.17586634, 0.0062733707, 0.03847...",0.428181,0.453869,0.808209,A,False,C,False
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,3,0.40,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,...,"[0.037406296, 0.11757571, 0.023855807, 0.08883...","[0.10099166, 0.1493785, 0.084694505, 0.0487238...","[0.030623527, 0.19797918, 0.10676692, 0.043068...",0.672166,0.781778,0.761080,C,False,A,True
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,4,0.61,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,...,"[-0.029311944, 0.10550189, 0.05036179, -0.0101...","[0.017343253, 0.35315275, 0.20922275, -0.09720...","[-0.010322735, 0.35458034, 0.16574, 0.0417216,...",0.554718,0.669521,0.872565,B,True,B,True
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,5,0.60,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,...,"[0.04814419, 0.08036936, 0.0690699, -0.0082249...","[-0.1161132, 0.0615291, -0.0451972, -0.0884399...","[0.0076814005, 0.03361294, -0.0035526846, -0.0...",0.460525,0.812388,0.539009,E,True,A,False


---

## 7.1. Embeddings de 300 Dimensões


In [3]:
# Coletando os dados
X_300 = [
    np.array(embedding) for embedding in enem_data["enunciado_embbedings_word2vec_300"]
]  # Somente Embeddings 300

y_300 = enem_data["dificuldade"]  # Parâmetro de Dificuldade (B)

In [4]:
# Aplicando Transformações
add_list = [(y_300.min() * (-1)) + 1] * len(y_300)
y_300 = y_300 + add_list

# Aplicando Box-Cox
y_300, best_lambda = stats.boxcox(y_300)
print(best_lambda)

0.6122671791275521


### 7.1.1. Regressão Linear


In [5]:
# Separando em conjunto de treino e teste
X_train_300, X_test_300, y_train_300, y_test_300 = train_test_split(
    X_300, y_300, test_size=0.3, random_state=42
)

In [6]:
# Adicionando constante
X_train_300 = sm.add_constant(X_train_300)
X_test_300 = sm.add_constant(X_test_300)

In [7]:
# Criando o modelo e realizando predição
linear_model_300 = sm.OLS(y_train_300, X_train_300).fit()
prediction_linear_model_300 = linear_model_300.predict(X_test_300)

In [8]:
# Cálculo do RMSE
rmse_300 = root_mean_squared_error(y_test_300, prediction_linear_model_300)
print("RMSE: ", rmse_300)

RMSE:  0.9761726951842663


In [9]:
# Visualização do modelo e resultados
print(linear_model_300.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.749
Model:                            OLS   Adj. R-squared:                  0.131
Method:                 Least Squares   F-statistic:                     1.212
Date:                Mon, 23 Jun 2025   Prob (F-statistic):              0.111
Time:                        12:22:11   Log-Likelihood:                -32.417
No. Observations:                 423   AIC:                             666.8
Df Residuals:                     122   BIC:                             1885.
Df Model:                         300                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.4370      0.879      1.635      0.1

### 7.1.2. Regressão Lasso


In [10]:
# Separando em conjunto de treino e teste
X_train_300_lasso, X_test_300_lasso, y_train_300_lasso, y_test_300_lasso = (
    train_test_split(X_300, y_300, test_size=0.3, random_state=42)
)

In [11]:
lasso_model_300 = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train_300_lasso, y_train_300_lasso
)
pred_lasso_model_300 = lasso_model_300.predict(X_test_300_lasso)

print("Melhor R2 Score:", lasso_model_300.score(X_train_300_lasso, y_train_300_lasso))
print("Melhor Alpha:", lasso_model_300.alpha_)
print(
    "RMSE com o alpha escolhido:",
    root_mean_squared_error(y_test_300_lasso, pred_lasso_model_300),
)

Melhor R2 Score: 0.2943528462555647
Melhor Alpha: 0.001
RMSE com o alpha escolhido: 0.5107711437163239


### 7.1.3. Regressões com Corte (-3,3)


In [12]:
# Aplicando corte de dificuldade sugerido pelo artigo motivador (-3, 3)
enem_filtered = enem_data.copy()

enem_filtered = enem_filtered[
    (enem_filtered["dificuldade"] >= -3) & (enem_filtered["dificuldade"] <= 3)
]
enem_filtered["dificuldade"].describe()

count    591.000000
mean       1.281414
std        0.768053
min       -1.800920
25%        0.777885
50%        1.372000
75%        1.819295
max        2.992980
Name: dificuldade, dtype: float64

In [13]:
# Coletando os dados
X_300_filtered = [
    np.array(embedding)
    for embedding in enem_filtered["enunciado_embbedings_word2vec_300"]
]
y_300_filtered = enem_filtered["dificuldade"]

In [14]:
# Aplicando Transformações
add_list = [(y_300_filtered.min() * (-1)) + 1] * len(y_300_filtered)
y_300_filtered = y_300_filtered + add_list

# Aplicando Boxcox
y_300_filtered, best_lambda = stats.boxcox(y_300_filtered)
print(best_lambda)

1.763356176859512


### 7.1.3.A) Regressão Linear


In [15]:
# Dividindo os dados em treino e teste
X_train_300_filtered, X_test_300_filtered, y_train_300_filtered, y_test_300_filtered = (
    train_test_split(X_300_filtered, y_300_filtered, test_size=0.3, random_state=42)
)

In [16]:
# Adicionando constante
X_train_300_filtered = sm.add_constant(X_train_300_filtered)
X_test_300_filtered = sm.add_constant(X_test_300_filtered)

In [17]:
# Criando o modelo e realizando a predição
linear_model_300_filtered = sm.OLS(y_train_300_filtered, X_train_300_filtered).fit()
pred_linear_model_300_filtered = linear_model_300_filtered.predict(X_test_300_filtered)

In [18]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_300_filtered, pred_linear_model_300_filtered)
print("RMSE:", rms)

RMSE: 3.8836268218548113


In [19]:
print(linear_model_300_filtered.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.264
Method:                 Least Squares   F-statistic:                     1.492
Date:                Mon, 23 Jun 2025   Prob (F-statistic):            0.00718
Time:                        12:22:11   Log-Likelihood:                -584.18
No. Observations:                 413   AIC:                             1770.
Df Residuals:                     112   BIC:                             2981.
Df Model:                         300                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.4459      4.218      1.054      0.2

### 7.1.3.B) Regressão Lasso


In [20]:
# Dividindo os dados em treino e teste
(
    X_train_300_filtered_lasso,
    X_test_300_filtered_lasso,
    y_train_300_filtered_lasso,
    y_test_300_filtered_lasso,
) = train_test_split(X_300_filtered, y_300_filtered, test_size=0.3, random_state=42)

In [21]:
lasso_model_300_filtered = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100], random_state=0, tol=0.1
).fit(X_train_300_filtered_lasso, y_train_300_filtered_lasso)

pred_300_lasso_filtered = lasso_model_300_filtered.predict(X_test_300_filtered_lasso)

print(
    "Melhor R2 score:",
    lasso_model_300_filtered.score(
        X_train_300_filtered_lasso, y_train_300_filtered_lasso
    ),
)
print("Melhor Alpha:", lasso_model_300_filtered.alpha_)
print(
    "RMSE com o Alpha escolhido:",
    root_mean_squared_error(y_test_300_filtered_lasso, pred_300_lasso_filtered),
)

Melhor R2 score: 0.11916915027399466
Melhor Alpha: 0.01
RMSE com o Alpha escolhido: 1.9332350644870202


### 7.1.4. LLM & Similaridade Features


In [22]:
# Coletando os dados
y_300_all_features = enem_data["dificuldade"]
X_300_all_features = [
    np.concatenate(
        [
            np.array(embedding),
            [
                pc_amostra_acertos,
                acerto_deepseek,
                acerto_llamma,
                similaridade_enunciado_gabarito_300,
                similaridade_enunciado_distratores_300,
                similaridade_gabarito_distratores_300,
            ],
        ]
    )
    for embedding, pc_amostra_acertos, acerto_deepseek, acerto_llamma, similaridade_enunciado_gabarito_300, similaridade_enunciado_distratores_300, similaridade_gabarito_distratores_300 in zip(
        enem_data["enunciado_embbedings_word2vec_300"],
        enem_data["pc_amostra_acertos"],
        enem_data["acerto_deepseek"],
        enem_data["acerto_llamma"],
        enem_data["similaridade_enunciado_gabarito_300"],
        enem_data["similaridade_enunciado_distratores_300"],
        enem_data["similaridade_gabarito_distratores_300"],
    )
]

In [23]:
# Aplicando Transformações
add_list = [(y_300_all_features.min() * (-1)) + 1] * len(y_300_all_features)
y_300_all_features = y_300_all_features + add_list

# Aplicando Boxcox
y_300_all_features, best_lambda = stats.boxcox(y_300_all_features)
print(best_lambda)

0.6122671791275521


### 7.1.4.A) Regressão linear


In [24]:
# Separando em conjunto de treino e teste

(
    X_train_300_all_features,
    X_test_300_all_features,
    y_train_300_all_features,
    y_test_300_all_features,
) = train_test_split(
    X_300_all_features, y_300_all_features, test_size=0.3, random_state=42
)

In [25]:
# Adicionando constante
X_train_300_all_features = sm.add_constant(X_train_300_all_features)
X_test_300_all_features = sm.add_constant(X_test_300_all_features)

In [26]:
# Criando o modelo e realizando predição
linear_model_300_all_features = sm.OLS(
    y_train_300_all_features, X_train_300_all_features
).fit()
pred_linear_model_300_all_features = linear_model_300_all_features.predict(
    X_test_300_all_features
)

In [27]:
# Cálculo do RMSE
rms = root_mean_sq0uared_error(y_test_300_all_features, pred_linear_model_300_all_features)
print("RMSE", rms)

RMSE 0.8435039183617573


In [28]:
print(linear_model_300_all_features.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.816
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     1.680
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           0.000689
Time:                        12:22:11   Log-Likelihood:                 33.429
No. Observations:                 423   AIC:                             547.1
Df Residuals:                     116   BIC:                             1790.
Df Model:                         306                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.6139      0.853      3.065      0.0

### 7.1.4.B) Lasso


In [29]:
# Separando em conjunto de treino e teste
(
    X_train_300_lasso_all_features,
    X_test_300_lasso_all_features,
    y_train_300_lasso_all_features,
    y_test_300_lasso_all_features,
) = train_test_split(
    X_300_all_features, y_300_all_features, test_size=0.3, random_state=42
)

In [30]:
lasso_model_all_features_300 = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0
).fit(X_train_300_lasso_all_features, y_train_300_lasso_all_features)

pred_300_lasso_all_features = lasso_model_all_features_300.predict(
    X_test_300_lasso_all_features
)

print(
    "melhor r2 score:",
    lasso_model_all_features_300.score(
        X_train_300_lasso_all_features, y_train_300_lasso_all_features
    ),
)
print("melhor alpha:", lasso_model_all_features_300.alpha_)
print(
    "RMSE com o alpha escolhido:",
    root_mean_squared_error(y_test_300_lasso_all_features, pred_300_lasso_all_features),
)

melhor r2 score: 0.44613602576130407
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.43606393860880777


---

## 7.2. Embeddings de 100 Dimensões


In [31]:
# Coletando os dados
X_100 = [
    np.array(embedding) for embedding in enem_data["enunciado_embbedings_word2vec_100"]
]  # Somente Embeddings

y_100 = enem_data["dificuldade"]  # Parâmetro de Dificuldade (B)

In [32]:
# Aplicando Transformações
add_list = [(y_100.min() * (-1)) + 1] * len(y_100)
y_100 = y_100 + add_list

# Aplicando Box-Cox
y_100, best_lambda = stats.boxcox(y_100)
print(best_lambda)

0.6122671791275521


### 7.2.1. Regressão Linear


In [33]:
# Separando em conjunto de treino e teste
X_train_100, X_test_100, y_train_100, y_test_100 = train_test_split(
    X_100, y_100, test_size=0.3, random_state=42
)

In [34]:
# Adicionando constante
X_train_100 = sm.add_constant(X_train_100)
X_test_100 = sm.add_constant(X_test_100)

In [35]:
# Criando o modelo e realizando predição
linear_model_100 = sm.OLS(y_train_100, X_train_100).fit()
pred_linear_model_100 = linear_model_100.predict(X_test_100)

In [36]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_100, pred_linear_model_100)
print("RMSE: ", rms)

RMSE:  0.585527171655945


In [37]:
# Visualização do modelo e resultados
print(linear_model_100.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.339
Model:                            OLS   Adj. R-squared:                  0.134
Method:                 Least Squares   F-statistic:                     1.654
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           0.000550
Time:                        12:22:12   Log-Likelihood:                -236.82
No. Observations:                 423   AIC:                             675.6
Df Residuals:                     322   BIC:                             1084.
Df Model:                         100                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0246      0.444      4.561      0.0

### 7.2.2. Regressão Lasso


In [38]:
# Separando em conjunto de treino e teste
X_train_100_lasso, X_test_100_lasso, y_train_100_lasso, y_test_100_lasso = (
    train_test_split(X_100, y_100, test_size=0.3, random_state=42)
)

In [39]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train_100_lasso, y_train_100_lasso
)
pred_lasso = model_lasso.predict(X_test_100_lasso)

print("Melhor R2 Score:", model_lasso.score(X_train_100_lasso, y_train_100_lasso))
print("Melhor Alpha:", model_lasso.alpha_)
print(
    "RMSE com o alpha escolhido:", root_mean_squared_error(y_test_100_lasso, pred_lasso)
)

Melhor R2 Score: 0.19736162286851588
Melhor Alpha: 0.001
RMSE com o alpha escolhido: 0.5076129610285552


### 7.2.3. Regressões com Corte (-3,3)


In [40]:
# Aplicando corte de dificuldade sugerido pelo artigo motivador (-3, 3)
enem_filtered = enem_data.copy()

enem_filtered = enem_filtered[
    (enem_filtered["dificuldade"] >= -3) & (enem_filtered["dificuldade"] <= 3)
]
enem_filtered["dificuldade"].describe()

count    591.000000
mean       1.281414
std        0.768053
min       -1.800920
25%        0.777885
50%        1.372000
75%        1.819295
max        2.992980
Name: dificuldade, dtype: float64

In [41]:
# Coletando os dados
X_100_filtered = [
    np.array(embedding)
    for embedding in enem_filtered["enunciado_embbedings_word2vec_100"]
]
y_100_filtered = enem_filtered["dificuldade"]

In [42]:
# Aplicando Transformações
add_list = [(y_100_filtered.min() * (-1)) + 1] * len(y_100_filtered)
y_100_filtered = y_100_filtered + add_list

# Aplicando Boxcox
y_100_filtered, best_lambda = stats.boxcox(y_100_filtered)
print(best_lambda)

1.763356176859512


### 7.2.3.A) Regressão Linear


In [43]:
# Dividindo os dados em treino e teste
X_train_100_filtered, X_test_100_filtered, y_train_100_filtered, y_test_100_filtered = (
    train_test_split(X_100_filtered, y_100_filtered, test_size=0.3, random_state=42)
)

In [44]:
# Adicionando constante
X_train_100_filtered = sm.add_constant(X_train_100_filtered)
X_test_100_filtered = sm.add_constant(X_test_100_filtered)

In [45]:
# Criando o modelo e realizando a predição
linear_model_100_filtered = sm.OLS(y_train_100_filtered, X_train_100_filtered).fit()
pred_linear_model_100_filtered = linear_model_100_filtered.predict(X_test_100_filtered)

In [46]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_100_filtered, pred_linear_model_100_filtered)
print("RMSE:", rms)

RMSE: 2.367315418677431


In [47]:
print(linear_model_100_filtered.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.364
Model:                            OLS   Adj. R-squared:                  0.161
Method:                 Least Squares   F-statistic:                     1.788
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           8.32e-05
Time:                        12:22:12   Log-Likelihood:                -822.86
No. Observations:                 413   AIC:                             1848.
Df Residuals:                     312   BIC:                             2254.
Df Model:                         100                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.3421      1.987      3.192      0.0

### 7.2.3.B) Regressão Lasso


In [48]:
# Dividindo os dados em treino e teste
(
    X_train_100_filtered_lasso,
    X_test_100_filtered_lasso,
    y_train_100_filtered_lasso,
    y_test_100_filtered_lasso,
) = train_test_split(X_100_filtered, y_100_filtered, test_size=0.3, random_state=42)

In [49]:
lasso_model_100_filtered = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100], random_state=0, tol=0.1
).fit(X_train_100_filtered_lasso, y_train_100_filtered_lasso)

pred_lasso_filtered_100 = lasso_model_100_filtered.predict(X_test_100_filtered_lasso)

print(
    "Melhor R2 score:",
    lasso_model_100_filtered.score(
        X_train_100_filtered_lasso, y_train_100_filtered_lasso
    ),
)
print("Melhor Alpha:", lasso_model_100_filtered.alpha_)
print(
    "RMSE com o Alpha escolhido:",
    root_mean_squared_error(y_test_100_filtered_lasso, pred_lasso_filtered_100),
)

Melhor R2 score: 0.10203652396417562
Melhor Alpha: 0.01
RMSE com o Alpha escolhido: 1.9622476103150315


### 7.2.4. LLM & Similaridade Features


In [50]:
# Coletando os dados
y_100_all_features = enem_data["dificuldade"]
X_100_all_features = [
    np.concatenate(
        [
            np.array(embedding),
            [
                pc_amostra_acertos,
                acerto_deepseek,
                acerto_llamma,
                similaridade_enunciado_gabarito_100,
                similaridade_enunciado_distratores_100,
                similaridade_gabarito_distratores_100,
            ],
        ]
    )
    for embedding, pc_amostra_acertos, acerto_deepseek, acerto_llamma, similaridade_enunciado_gabarito_100, similaridade_enunciado_distratores_100, similaridade_gabarito_distratores_100 in zip(
        enem_data["enunciado_embbedings_word2vec_100"],
        enem_data["pc_amostra_acertos"],
        enem_data["acerto_deepseek"],
        enem_data["acerto_llamma"],
        enem_data["similaridade_enunciado_gabarito_100"],
        enem_data["similaridade_enunciado_distratores_100"],
        enem_data["similaridade_gabarito_distratores_100"],
    )
]

In [51]:
# Aplicando Transformações
add_list = [(y_100_all_features.min() * (-1)) + 1] * len(y_100_all_features)
y_100_all_features = y_100_all_features + add_list

# Aplicando Boxcox
y_100_all_features, best_lambda = stats.boxcox(y_100_all_features)
print(best_lambda)

0.6122671791275521


### 7.2.4.A) Regressão linear


In [52]:
# Separando em conjunto de treino e teste

(
    X_train_100_all_features,
    X_test_100_all_features,
    y_train_100_all_features,
    y_test_100_all_features,
) = train_test_split(
    X_100_all_features, y_100_all_features, test_size=0.3, random_state=42
)

In [53]:
# Adicionando constante
X_train_100_all_features = sm.add_constant(X_train_100_all_features)
X_test_100_all_features = sm.add_constant(X_test_100_all_features)

In [54]:
# Criando o modelo e realizando predição
linear_model_100_all_features = sm.OLS(
    y_train_100_all_features, X_train_100_all_features
).fit()
pred_linear_model_100_all_features = linear_model_100_all_features.predict(
    X_test_100_all_features
)

In [55]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_100_all_features, pred_linear_model_100_all_features)
print("RMSE", rms)

RMSE 0.5168251798138981


In [56]:
print(linear_model_100_all_features.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.506
Model:                            OLS   Adj. R-squared:                  0.341
Method:                 Least Squares   F-statistic:                     3.058
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           1.60e-14
Time:                        12:22:12   Log-Likelihood:                -175.20
No. Observations:                 423   AIC:                             564.4
Df Residuals:                     316   BIC:                             997.5
Df Model:                         106                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.8745      0.433      6.631      0.0

### 7.2.4.B) Lasso


In [57]:
# Separando em conjunto de treino e teste
(
    X_train_100_lasso_all_features,
    X_test_100_lasso_all_features,
    y_train_100_lasso_all_features,
    y_test_100_lasso_all_features,
) = train_test_split(
    X_100_all_features, y_100_all_features, test_size=0.3, random_state=42
)

In [58]:
lasso_model_all_features_100 = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0
).fit(X_train_100_lasso_all_features, y_train_100_lasso_all_features)

pred_100_lasso_all_features = lasso_model_all_features_100.predict(
    X_test_100_lasso_all_features
)

print(
    "melhor r2 score:",
    lasso_model_all_features_100.score(
        X_train_100_lasso_all_features, y_train_100_lasso_all_features
    ),
)
print("melhor alpha:", lasso_model_all_features_100.alpha_)
print(
    "RMSE com o alpha escolhido:",
    root_mean_squared_error(y_test_100_lasso_all_features, pred_100_lasso_all_features),
)

melhor r2 score: 0.389775244502918
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.4310506662940833


---

## 7.3. Embeddings de 50 Dimensões


In [59]:
# Coletando os dados
X_50 = [
    np.array(embedding) for embedding in enem_data["enunciado_embbedings_word2vec_50"]
]  # Somente Embeddings 50

y_50 = enem_data["dificuldade"]  # Parâmetro de Dificuldade (B)

In [60]:
# Aplicando Transformações
add_list = [(y_50.min() * (-1)) + 1] * len(y_50)
y_50 = y_50 + add_list

# Aplicando Box-Cox
y_50, best_lambda = stats.boxcox(y_50)
print(best_lambda)

0.6122671791275521


### 7.3.1. Regressão Linear


In [61]:
# Separando em conjunto de treino e teste
X_train_50, X_test_50, y_train_50, y_test_50 = train_test_split(
    X_50, y_50, test_size=0.3, random_state=42
)

In [62]:
# Adicionando constante
X_train_50 = sm.add_constant(X_train_50)
X_test_50 = sm.add_constant(X_test_50)

In [63]:
# Criando o modelo e realizando predição
linear_model_50 = sm.OLS(y_train_50, X_train_50).fit()
pred_linear_model_50 = linear_model_50.predict(X_test_50)

In [64]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_50, pred_linear_model_50)
print("RMSE: ", rms)

RMSE:  0.535776637121338


In [65]:
# Visualização do modelo e resultados
print(linear_model_50.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.221
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     2.109
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           4.96e-05
Time:                        12:22:12   Log-Likelihood:                -271.73
No. Observations:                 423   AIC:                             645.5
Df Residuals:                     372   BIC:                             851.9
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7124      0.363      4.714      0.0

### 7.3.2. Regressão Lasso


In [66]:
# Separando em conjunto de treino e teste
X_train_50_lasso, X_test_50_lasso, y_train_50_lasso, y_test_50_lasso = train_test_split(
    X_50, y_50, test_size=0.3, random_state=42
)

In [67]:
lasso_model_50 = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train_50_lasso, y_train_50_lasso
)
pred_lasso_model_50 = lasso_model_50.predict(X_test_50_lasso)

print("Melhor R2 Score:", lasso_model_50.score(X_train_50_lasso, y_train_50_lasso))
print("Melhor Alpha:", lasso_model_50.alpha_)
print(
    "RMSE com o alpha escolhido:",
    root_mean_squared_error(y_test_50_lasso, pred_lasso_model_50),
)

Melhor R2 Score: 0.15636826114743163
Melhor Alpha: 0.001
RMSE com o alpha escolhido: 0.49882607650816907


### 7.3.3. Regressões com Corte (-3,3)


In [68]:
# Aplicando corte de dificuldade sugerido pelo artigo motivador (-3, 3)
enem_filtered = enem_data.copy()

enem_filtered = enem_filtered[
    (enem_filtered["dificuldade"] >= -3) & (enem_filtered["dificuldade"] <= 3)
]
enem_filtered["dificuldade"].describe()

count    591.000000
mean       1.281414
std        0.768053
min       -1.800920
25%        0.777885
50%        1.372000
75%        1.819295
max        2.992980
Name: dificuldade, dtype: float64

In [69]:
# Coletando os dados
X_50_filtered = [
    np.array(embedding)
    for embedding in enem_filtered["enunciado_embbedings_word2vec_50"]
]
y_50_filtered = enem_filtered["dificuldade"]

In [70]:
# Aplicando Transformações
add_list = [(y_50_filtered.min() * (-1)) + 1] * len(y_50_filtered)
y_50_filtered = y_50_filtered + add_list

# Aplicando Boxcox
y_50_filtered, best_lambda = stats.boxcox(y_50_filtered)
print(best_lambda)

1.763356176859512


### 7.3.3.A) Regressão Linear


In [71]:
# Dividindo os dados em treino e teste
X_train_50_filtered, X_test_50_filtered, y_train_50_filtered, y_test_50_filtered = (
    train_test_split(X_50_filtered, y_50_filtered, test_size=0.3, random_state=42)
)

In [72]:
# Adicionando constante
X_train_50_filtered = sm.add_constant(X_train_50_filtered)
X_test_50_filtered = sm.add_constant(X_test_50_filtered)

In [73]:
# Criando o modelo e realizando a predição
linear_model_50_filtered = sm.OLS(y_train_50_filtered, X_train_50_filtered).fit()
pred_linear_model_50_filtered = linear_model_50_filtered.predict(X_test_50_filtered)

In [74]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_50_filtered, pred_linear_model_50_filtered)
print("RMSE:", rms)

RMSE: 2.012211560671998


In [75]:
print(linear_model_50_filtered.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.222
Model:                            OLS   Adj. R-squared:                  0.115
Method:                 Least Squares   F-statistic:                     2.070
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           7.89e-05
Time:                        12:22:12   Log-Likelihood:                -864.50
No. Observations:                 413   AIC:                             1831.
Df Residuals:                     362   BIC:                             2036.
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.8044      1.654      3.508      0.0

### 7.3.3.B) Regressão Lasso


In [76]:
# Dividindo os dados em treino e teste
(
    X_train_50_filtered_lasso,
    X_test_50_filtered_lasso,
    y_train_50_filtered_lasso,
    y_test_50_filtered_lasso,
) = train_test_split(X_50_filtered, y_50_filtered, test_size=0.3, random_state=42)

In [77]:
lasso_model_50_filtered = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100], random_state=0, tol=0.1
).fit(X_train_50_filtered_lasso, y_train_50_filtered_lasso)

pred_lasso_model_50_filtered = lasso_model_50_filtered.predict(X_test_50_filtered_lasso)

print(
    "Melhor R2 score:",
    lasso_model_50_filtered.score(X_train_50_filtered_lasso, y_train_50_filtered_lasso),
)
print("Melhor Alpha:", lasso_model_50_filtered.alpha_)
print(
    "RMSE com o Alpha escolhido:",
    root_mean_squared_error(y_test_50_filtered_lasso, pred_lasso_model_50_filtered),
)

Melhor R2 score: 0.11326315996760328
Melhor Alpha: 0.01
RMSE com o Alpha escolhido: 1.9593889922740213


### 7.3.4. LLM & Similaridade Features


In [78]:
# Coletando os dados
y_50_all_features = enem_data["dificuldade"]
X_50_all_features = [
    np.concatenate(
        [
            np.array(embedding),
            [
                pc_amostra_acertos,
                acerto_deepseek,
                acerto_llamma,
                similaridade_enunciado_gabarito_50,
                similaridade_enunciado_distratores_50,
                similaridade_gabarito_distratores_50,
            ],
        ]
    )
    for embedding, pc_amostra_acertos, acerto_deepseek, acerto_llamma, similaridade_enunciado_gabarito_50, similaridade_enunciado_distratores_50, similaridade_gabarito_distratores_50 in zip(
        enem_data["enunciado_embbedings_word2vec_50"],
        enem_data["pc_amostra_acertos"],
        enem_data["acerto_deepseek"],
        enem_data["acerto_llamma"],
        enem_data["similaridade_enunciado_gabarito_50"],
        enem_data["similaridade_enunciado_distratores_50"],
        enem_data["similaridade_gabarito_distratores_50"],
    )
]

In [79]:
# Aplicando Transformações
add_list = [(y_50_all_features.min() * (-1)) + 1] * len(y_50_all_features)
y_50_all_features = y_50_all_features + add_list

# Aplicando Boxcox
y_50_all_features, best_lambda = stats.boxcox(y_50_all_features)
print(best_lambda)

0.6122671791275521


### 7.3.4.A) Regressão linear


In [80]:
# Separando em conjunto de treino e teste

(
    X_train_50_all_features,
    X_test_50_all_features,
    y_train_50_all_features,
    y_test_50_all_features,
) = train_test_split(
    X_50_all_features, y_50_all_features, test_size=0.3, random_state=42
)

In [81]:
# Adicionando constante
X_train_50_all_features = sm.add_constant(X_train_50_all_features)
X_test_50_all_features = sm.add_constant(X_test_50_all_features)

In [82]:
# Criando o modelo e realizando predição
linear_model_50_all_features = sm.OLS(
    y_train_50_all_features, X_train_50_all_features
).fit()
pred_linear_model_50_all_features = linear_model_50_all_features.predict(
    X_test_50_all_features
)

In [83]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test_50_all_features, pred_linear_model_50_all_features)
print("RMSE", rms)

RMSE 0.46476083299843


In [84]:
print(linear_model_50_all_features.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.423
Model:                            OLS   Adj. R-squared:                  0.335
Method:                 Least Squares   F-statistic:                     4.795
Date:                Mon, 23 Jun 2025   Prob (F-statistic):           1.54e-20
Time:                        12:22:13   Log-Likelihood:                -208.14
No. Observations:                 423   AIC:                             530.3
Df Residuals:                     366   BIC:                             761.0
Df Model:                          56                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.4552      0.370      6.628      0.0

### 7.3.4.B) Lasso


In [85]:
# Separando em conjunto de treino e teste
(
    X_train_50_lasso_all_features,
    X_test_50_lasso_all_features,
    y_train_50_lasso_all_features,
    y_test_50_lasso_all_features,
) = train_test_split(
    X_50_all_features, y_50_all_features, test_size=0.3, random_state=42
)

In [86]:
lasso_model_all_features_50 = LassoCV(
    alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0
).fit(X_train_50_lasso_all_features, y_train_50_lasso_all_features)

pred_50_lasso_all_features = lasso_model_all_features_50.predict(
    X_test_50_lasso_all_features
)

print(
    "melhor r2 score:",
    lasso_model_all_features_50.score(
        X_train_50_lasso_all_features, y_train_50_lasso_all_features
    ),
)
print("melhor alpha:", lasso_model_all_features_50.alpha_)
print(
    "RMSE com o alpha escolhido:",
    root_mean_squared_error(y_test_50_lasso_all_features, pred_50_lasso_all_features),
)

melhor r2 score: 0.36036203341987594
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.41969413512004844


---
